In [1]:
import io
import os
import time

import Levenshtein
from PIL import Image, ImageEnhance
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from msrest.authentication import CognitiveServicesCredentials
from spellchecker import SpellChecker

'''
Authenticate
Authenticates your credentials and creates a client.
'''
subscription_key = os.environ["VISION_KEY"]
endpoint = os.environ["VISION_ENDPOINT"]
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
'''
END - Authenticate
'''

# img = open("data/images/test1.png", "rb")
img = open("data/images/test2.jpeg", "rb")
read_response = computervision_client.read_in_stream(
    image=img,
    mode="Printed",
    raw=True
)
# print(read_response.as_dict())

operation_id = read_response.headers['Operation-Location'].split('/')[-1]
while True:
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)

# Print the detected text, line by line
result = []
if read_result.status == OperationStatusCodes.succeeded:
    for text_result in read_result.analyze_result.read_results:
        for line in text_result.lines:
            # print(line.text)
            result.append(line.text)


recognized_text = " ".join(result)
recognized_words = recognized_text.split()
print(recognized_text)
ground_truth_text = "Succes în rezolvarea tEMELOR la LABORAtoarele de Inteligență Aritificială!"
ground_truth_words = ground_truth_text.split()

Lucces in resolvarea TEMELOR la LABORA toarele de Inteligenta Artificialà!


In [2]:
def cerinta3():
    img = Image.open("data/images/blurred2.jpg")
    enhancer = ImageEnhance.Contrast(img)
    img_contrast = enhancer.enhance(2) # Imbunatatim contrastul imaginii pentru a imbunatati recunoasterea textului scris de mana
    img_gray = img_contrast.convert("L") # Convertim imaginea in tonuri de gri pentru a imbunatati recunoasterea textului scris de mana
    buffer = io.BytesIO()
    img_gray.save(buffer, format="JPEG")
    buffer.seek(0)

    read_response = computervision_client.read_in_stream(
        image=buffer,
        mode="Handwritten",
        raw=True,
        language = "en" # setam limba romana pentru a imbunatati recunoasterea textului scris de mana in limba romana
    )

    operation_id = read_response.headers['Operation-Location'].split('/')[-1]
    while True:
        read_result = computervision_client.get_read_result(operation_id)
        if read_result.status not in ['notStarted', 'running']:
            break
        time.sleep(1)

    # Print the detected text, line by line
    result = []
    if read_result.status == OperationStatusCodes.succeeded:
        for text_result in read_result.analyze_result.read_results:
            for line in text_result.lines:
                # print(line.text)
                result.append(line.text)

    spell = SpellChecker(language='en') # Incarcam un dictionar de cuvinte in limba engleza pentru a imbunatati corectarea cuvintelor recunoscute gresit
    corrected_words = []
    for word in result:
        corrected = spell.correction(word)
        if corrected is None:
            corrected_words.append(word)
        else:
            corrected_words.append(corrected)
    recognized_text = " ".join(corrected_words)
    print("Good luck solving the Artificial Intelligence labs!")
    print(recognized_text)

In [3]:
cerinta3()

Good luck solving the Artificial Intelligence labs!
Good luck soloing the Atfoial Intelligence Rates !
